In [10]:
import re
from pathlib import Path
from difflib import SequenceMatcher

# CORRECT path (Kaggle nested under /datasets/<username>/)
COMMERCIAL_ROOT = Path("/kaggle/input/datasets/shantanuvedanteog/commercial-raw-videos/commerical_video_raw")
OUTPUT_ROOT = Path("/kaggle/working/commercial_renamed")
OUTPUT_ROOT.mkdir(exist_ok=True)

PROMPTS_FILE = Path("/kaggle/input/datasets/shantanuvedanteog/week2-prompts-v1generation-txt/week2_prompts_v1(generation).txt")

with open(PROMPTS_FILE) as f:
    prompt_lines = [line.strip() for line in f 
                    if line.strip() and not line.strip().startswith("#") and "|" in line]

prompts_lookup = {}
for line in prompt_lines:
    prompt_id, category, prompt_text = line.split("|", 2)
    prompts_lookup[prompt_id] = {
        "category": category,
        "prompt_text": prompt_text,
        "prompt_lower": prompt_text.lower(),
    }

print(f"Loaded {len(prompts_lookup)} prompts")

# Verify commercial folders
print("\n--- Commercial folders ---")
for name in ["kling3.0", "gemini_omni_flash", "seedance2.0"]:
    path = COMMERCIAL_ROOT / name
    if path.exists():
        count = len(list(path.glob("*.mp4")))
        print(f"  {name}: {count} mp4 files")
    else:
        print(f"  {name}: NOT FOUND")

Loaded 40 prompts

--- Commercial folders ---
  kling3.0: 32 mp4 files
  gemini_omni_flash: 32 mp4 files
  seedance2.0: 32 mp4 files


In [11]:
def extract_prompt_from_seedance(filename):
    """
    Seedance pattern: "A person applying makeup-seedance2.0.mp4"
    Strip -seedance* suffix.
    """
    stem = Path(filename).stem
    # Remove -seedance suffix (with any version number)
    text = re.sub(r'\s*-\s*seedance[\d.\s]*$', '', stem, flags=re.IGNORECASE)
    return text.strip()


def extract_prompt_from_gemini(filename):
    """
    Gemini Omni Flash pattern: "A cat stretching lazily -gemini_omni_flash.mp4"
    Strip -gemini* suffix.
    """
    stem = Path(filename).stem
    text = re.sub(r'\s*-\s*gemini[\w_]*$', '', stem, flags=re.IGNORECASE)
    return text.strip()


def extract_prompt_from_kling(filename):
    """
    Kling pattern: "kling_20260721_VIDEO_A_family.mp4"
    Strip kling_<date>_VIDEO_ prefix, replace underscores with spaces.
    """
    stem = Path(filename).stem
    # Try common patterns
    # Pattern 1: kling_YYYYMMDD_VIDEO_<prompt>
    match = re.match(r'kling_\d+_VIDEO_(.+)', stem, flags=re.IGNORECASE)
    if match:
        text = match.group(1).replace('_', ' ')
        return text.strip()
    # Pattern 2: kling_<prompt>
    match = re.match(r'kling[\w_]*_(.+)', stem, flags=re.IGNORECASE)
    if match:
        text = match.group(1).replace('_', ' ')
        return text.strip()
    return stem.replace('_', ' ').strip()


# Test the parsers on your example filenames
test_cases = [
    ("A person applying makeup-seedance2.0.mp4", extract_prompt_from_seedance),
    ("A cat stretching lazily -gemini_omni_flash.mp4", extract_prompt_from_gemini),
    ("kling_20260721_VIDEO_A_family.mp4", extract_prompt_from_kling),
]

for filename, fn in test_cases:
    print(f"'{filename}' → '{fn(filename)}'")

'A person applying makeup-seedance2.0.mp4' → 'A person applying makeup'
'A cat stretching lazily -gemini_omni_flash.mp4' → 'A cat stretching lazily'
'kling_20260721_VIDEO_A_family.mp4' → 'A family'


In [12]:
def match_to_prompt(extracted_text, prompts_lookup, threshold=0.5):
    """
    Find the best matching prompt_id for extracted filename text.
    Uses prefix matching first (most reliable), then fuzzy fallback.
    Returns (prompt_id, confidence, prompt_text) or (None, 0, None).
    """
    if not extracted_text:
        return (None, 0, None)
    
    extracted_lower = extracted_text.lower().strip()
    extracted_words = extracted_lower.split()
    
    if not extracted_words:
        return (None, 0, None)
    
    # Strategy 1: prefix match (best when filename is truncation of prompt)
    prefix_matches = []
    for pid, data in prompts_lookup.items():
        if data["prompt_lower"].startswith(extracted_lower):
            prefix_matches.append((pid, 1.0, data["prompt_text"]))
    
    if len(prefix_matches) == 1:
        return prefix_matches[0]
    
    # Strategy 2: word-level prefix (first N words of extracted match first N words of prompt)
    word_matches = []
    for pid, data in prompts_lookup.items():
        prompt_words = data["prompt_lower"].split()
        # How many leading words match?
        match_count = 0
        for e_word, p_word in zip(extracted_words, prompt_words):
            if e_word == p_word:
                match_count += 1
            else:
                break
        if match_count >= 2:  # at least 2 leading words match
            score = match_count / max(len(extracted_words), 3)
            word_matches.append((pid, score, data["prompt_text"], match_count))
    
    if word_matches:
        word_matches.sort(key=lambda x: (-x[3], -x[1]))  # most matched words wins
        best = word_matches[0]
        return (best[0], best[1], best[2])
    
    # Strategy 3: fuzzy match as fallback
    fuzzy_scores = []
    for pid, data in prompts_lookup.items():
        # Compare extracted against prompt start (truncated to same length as extracted)
        prompt_start = data["prompt_lower"][:len(extracted_lower)]
        ratio = SequenceMatcher(None, extracted_lower, prompt_start).ratio()
        fuzzy_scores.append((pid, ratio, data["prompt_text"]))
    
    fuzzy_scores.sort(key=lambda x: -x[1])
    if fuzzy_scores and fuzzy_scores[0][1] >= threshold:
        return fuzzy_scores[0]
    
    return (None, 0, None)


# Quick test
test_extracted = "A cat stretching lazily"
result = match_to_prompt(test_extracted, prompts_lookup)
print(f"Test: '{test_extracted}' → {result}")

Test: 'A cat stretching lazily' → ('w2_032', 1.0, 'A cat stretching lazily on a sunlit windowsill, indoor scene')


In [15]:
# Configuration per generator
generators_config = {
    "seedance": {
        "input_dir": COMMERCIAL_ROOT / "seedance2.0",
        "extractor": extract_prompt_from_seedance,
        "output_suffix": "seedance",
    },
    "kling": {
        "input_dir": COMMERCIAL_ROOT / "kling3.0",
        "extractor": extract_prompt_from_kling,
        "output_suffix": "kling",
    },
    "gemini_omni_flash": {
        "input_dir": COMMERCIAL_ROOT / "gemini_omni_flash",
        "extractor": extract_prompt_from_gemini,
        "output_suffix": "gemini_omni_flash",
    },
}

# Track everything before any renaming
rename_plan = []
unmatched = []
ambiguous = []

for gen_name, config in generators_config.items():
    input_dir = config["input_dir"]
    if not input_dir.exists():
        print(f"WARNING: {input_dir} does not exist, skipping {gen_name}")
        continue
    
    videos = sorted(input_dir.glob("*.mp4"))
    print(f"\n=== {gen_name}: {len(videos)} videos ===")
    
    for src in videos:
        extracted = config["extractor"](src.name)
        prompt_id, confidence, matched_prompt = match_to_prompt(extracted, prompts_lookup)
        
        if prompt_id is None:
            unmatched.append({
                "generator": gen_name,
                "filename": src.name,
                "extracted_text": extracted,
            })
            print(f"  [UNMATCHED] {src.name}")
            print(f"              extracted: '{extracted}'")
            continue
        
        new_name = f"{prompt_id}_{config['output_suffix']}.mp4"
        rename_plan.append({
            "generator": gen_name,
            "old_name": src.name,
            "new_name": new_name,
            "prompt_id": prompt_id,
            "confidence": confidence,
            "extracted": extracted,
            "matched_prompt": matched_prompt[:60] + "...",
        })

# Detect ambiguous (multiple files → same target name)
from collections import Counter
target_counts = Counter((p["generator"], p["new_name"]) for p in rename_plan)
duplicates = {k: v for k, v in target_counts.items() if v > 1}
if duplicates:
    print(f"\n⚠  {len(duplicates)} DUPLICATE TARGET NAMES:")
    for (gen, name), count in duplicates.items():
        print(f"  {gen}/{name} → {count} source files")
        for p in rename_plan:
            if p["generator"] == gen and p["new_name"] == name:
                print(f"    - {p['old_name']} (confidence {p['confidence']:.2f})")

# Summary
print(f"\n{'='*60}")
print(f"Total files planned: {len(rename_plan)}")
print(f"Unmatched: {len(unmatched)}")
print(f"Duplicates (need manual review): {len(duplicates)}")

# Show first 5 successful matches per generator
print("\nFirst 3 matches per generator (spot-check these):")
for gen_name in generators_config:
    gen_plan = [p for p in rename_plan if p["generator"] == gen_name][:3]
    print(f"\n  {gen_name}:")
    for p in gen_plan:
        print(f"    {p['old_name']}")
        print(f"      → {p['new_name']}  (extracted: '{p['extracted']}', conf: {p['confidence']:.2f})")

if unmatched:
    print(f"\n⚠  {len(unmatched)} UNMATCHED files — investigate before running Cell 5:")
    for u in unmatched:
        print(f"  [{u['generator']}] {u['filename']}")
        print(f"    extracted: '{u['extracted_text']}'")


=== seedance: 32 videos ===

=== kling: 32 videos ===

=== gemini_omni_flash: 32 videos ===

Total files planned: 96
Unmatched: 0
Duplicates (need manual review): 0

First 3 matches per generator (spot-check these):

  seedance:
    A cat stretching lazily-seedance2.0.mp4
      → w2_032_seedance.mp4  (extracted: 'A cat stretching lazily', conf: 1.00)
    A crowded market at night-seedance2.0.mp4
      → w2_039_seedance.mp4  (extracted: 'A crowded market at night', conf: 1.00)
    A dancer spinning in a red dress-seedance2.0.mp4
      → w2_037_seedance.mp4  (extracted: 'A dancer spinning in a red dress', conf: 1.00)

  kling:
    kling_20260721_VIDEO_A_family.mp4
      → w2_012_kling.mp4  (extracted: 'A family', conf: 1.00)
    kling_20260721_VIDEO_A_hand_pou.mp4
      → w2_007_kling.mp4  (extracted: 'A hand pou', conf: 1.00)
    kling_20260721_VIDEO_A_jogger.mp4
      → w2_017_kling.mp4  (extracted: 'A jogger', conf: 1.00)

  gemini_omni_flash:
    A cat stretching lazily -gemini_omni

In [16]:
from collections import Counter, defaultdict

# Check: does each generator have exactly 32 unique prompt_ids?
per_gen_prompt_ids = defaultdict(list)
for p in rename_plan:
    per_gen_prompt_ids[p["generator"]].append(p["prompt_id"])

print("=== Coverage check ===\n")
for gen, pids in per_gen_prompt_ids.items():
    unique = set(pids)
    dupes_in_gen = [pid for pid, count in Counter(pids).items() if count > 1]
    print(f"{gen}:")
    print(f"  Total files: {len(pids)}")
    print(f"  Unique prompt_ids: {len(unique)}")
    print(f"  Duplicates within generator: {dupes_in_gen if dupes_in_gen else 'none'}")
    print()

# Check: are all three generators covering the SAME 32 prompt_ids?
prompt_id_sets = {gen: set(pids) for gen, pids in per_gen_prompt_ids.items()}
all_gens = list(prompt_id_sets.keys())
if len(all_gens) >= 2:
    common = set.intersection(*[prompt_id_sets[g] for g in all_gens])
    print(f"Prompt_ids common to ALL {len(all_gens)} generators: {len(common)}")
    
    for gen in all_gens:
        only_here = prompt_id_sets[gen] - common
        if only_here:
            print(f"  {gen} has extra: {only_here}")

# Show which 32 prompts were used
print(f"\nAll 32 unique prompt_ids used:")
print(sorted(common))

# Show the low-confidence match specifically
print("\n=== Low-confidence matches (verify manually) ===")
for p in rename_plan:
    if p["confidence"] < 1.0:
        print(f"  [{p['generator']}] {p['old_name']}")
        print(f"    extracted: '{p['extracted']}'")
        print(f"    → {p['new_name']}")
        print(f"    matched prompt: {p['matched_prompt']}")
        print(f"    confidence: {p['confidence']:.2f}")

=== Coverage check ===

seedance:
  Total files: 32
  Unique prompt_ids: 32
  Duplicates within generator: none

kling:
  Total files: 32
  Unique prompt_ids: 32
  Duplicates within generator: none

gemini_omni_flash:
  Total files: 32
  Unique prompt_ids: 32
  Duplicates within generator: none

Prompt_ids common to ALL 3 generators: 32

All 32 unique prompt_ids used:
['w2_001', 'w2_002', 'w2_003', 'w2_004', 'w2_006', 'w2_007', 'w2_008', 'w2_009', 'w2_011', 'w2_012', 'w2_013', 'w2_014', 'w2_016', 'w2_017', 'w2_018', 'w2_019', 'w2_021', 'w2_022', 'w2_023', 'w2_024', 'w2_026', 'w2_027', 'w2_028', 'w2_029', 'w2_031', 'w2_032', 'w2_033', 'w2_034', 'w2_036', 'w2_037', 'w2_038', 'w2_039']

=== Low-confidence matches (verify manually) ===
  [kling] kling_20260721_VIDEO_Close_up_of_a_person_typing.mp4
    extracted: 'Close up of a person typing'
    → w2_006_kling.mp4
    matched prompt: Close-up of a person typing on a laptop keyboard, both hands...
    confidence: 0.96
  [kling] kling_202607

In [17]:
import shutil

# Copy (don't move) so originals are preserved
if unmatched:
    print(f"⚠  STOPPING: {len(unmatched)} unmatched files. Resolve before running.")
elif len(duplicates) > 0:
    print(f"⚠  STOPPING: {len(duplicates)} duplicate target names. Resolve before running.")
else:
    successful = 0
    for p in rename_plan:
        src = generators_config[p["generator"]]["input_dir"] / p["old_name"]
        gen_output_dir = OUTPUT_ROOT / p["generator"]
        gen_output_dir.mkdir(exist_ok=True)
        dst = gen_output_dir / p["new_name"]
        
        shutil.copy2(str(src), str(dst))
        successful += 1
        if successful % 20 == 0:
            print(f"  Renamed {successful}/{len(rename_plan)}")
    
    print(f"\nDone. {successful} files renamed and saved to {OUTPUT_ROOT}")
    print("\nBreakdown:")
    for gen in generators_config:
        count = len(list((OUTPUT_ROOT / gen).glob("*.mp4")))
        print(f"  {gen}: {count} videos")

  Renamed 20/96
  Renamed 40/96
  Renamed 60/96
  Renamed 80/96

Done. 96 files renamed and saved to /kaggle/working/commercial_renamed

Breakdown:
  seedance: 32 videos
  kling: 32 videos
  gemini_omni_flash: 32 videos


In [18]:
import json

# Save log so you can trace every rename
log_path = OUTPUT_ROOT / "rename_log.json"
with open(log_path, "w") as f:
    json.dump({
        "rename_plan": rename_plan,
        "unmatched": unmatched,
        "duplicates": [{"target": k, "count": v} for k, v in duplicates.items()],
    }, f, indent=2)

print(f"Log saved: {log_path}")

# Zip everything
shutil.make_archive("/kaggle/working/commercial_renamed", "zip", str(OUTPUT_ROOT))
!ls -la /kaggle/working/*.zip
print("\nDownload commercial_renamed.zip from Output panel.")

Log saved: /kaggle/working/commercial_renamed/rename_log.json
-rw-r--r-- 1 root root 339298993 Jul 23 09:47 /kaggle/working/commercial_renamed.zip

Download commercial_renamed.zip from Output panel.
